In [1]:
"""
BVAR Forecasting Model — Full Multivariate GLP Prior
======================================================
Specifications: BVAR(1), BVAR(AIC, p≤6).
Full multivariate Minnesota prior (GLP 2015). λ maximises log marginal
likelihood at each origin. Iterated multi-step. Expanding window from Feb 2012.
"""

import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from statsmodels.tsa.api import VAR
import warnings
warnings.filterwarnings("ignore")

# ── Parameters ────────────────────────────────────────────────────────────────
HORIZONS    = [1, 3, 6, 9, 12, 15, 18, 21, 24]
EVAL_START  = "2015-01-01"
VAR_START   = "2012-02-01"
AIC_LAG_MAX = 6
INPUT_FILE  = "VAR_Input_Data.xlsx"
OUTPUT_FILE = "Output_BVAR_forecasts.xlsx"

VARS = ["log_real_ttf", "hdd", "log_storage_avg", "log_lng_avg"]

SPECIFICATIONS = [
    ("BVAR(1)",        1,    False),
    ("BVAR(AIC,p<=6)", None, True),
]

# ── Load data ─────────────────────────────────────────────────────────────────
df = pd.read_excel(INPUT_FILE, sheet_name="Sheet1")
df.columns = ["date", "real_ttf", "log_real_ttf", "hdd", "log_storage_avg", "log_lng_avg"]
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

df_var = df[df["date"] >= VAR_START].dropna(subset=VARS).reset_index(drop=True)

# ── Helper: actual real price for a given year-month ─────────────────────────
def get_actual(ym_str):
    m = df[df["date"].dt.to_period("M").astype(str) == ym_str]
    return m["real_ttf"].values[0] if len(m) == 1 else np.nan

# ── Helper: build regressor matrix ───────────────────────────────────────────
def build_YX(data, p):
    T, K = data.shape
    Y = data[p:]
    X = np.ones((T - p, K * p + 1))
    for lag in range(1, p + 1):
        X[:, 1 + (lag - 1) * K : 1 + lag * K] = data[p - lag : T - lag]
    return Y, X

# ── Core: full multivariate GLP estimation ───────────────────────────────────
def bvar_glp(data, p):
    T, K = data.shape
    Y, X = build_YX(data, p)
    n    = Y.shape[0]                         # effective obs = T - p

    # ── Step 1: OLS Sigma (fixed throughout) ─────────────────────────────────
    B_ols   = np.linalg.lstsq(X, Y, rcond=None)[0]
    E_ols   = Y - X @ B_ols
    Sigma   = E_ols.T @ E_ols / n            # K x K residual covariance
    sigma2  = np.diag(Sigma)                  # equation-specific variances
    Sig_inv = np.linalg.inv(Sigma)
    XtX     = X.T @ X                         # (K*p+1) x (K*p+1)

    # ── Step 2: Prior builder ─────────────────────────────────────────────────
    def build_V0_diag(lam):

        v = []
        # Intercepts: one per equation, diffuse
        for k in range(K):
            v.append(1e6)
        # Lag coefficients: lag 1..p, equation k, variable j
        for lag in range(1, p + 1):
            for k in range(K):              # equation index
                for j in range(K):          # variable index
                    if j == k:
                        v.append((lam / lag) ** 2)
                    else:
                        v.append((lam / lag) ** 2 * sigma2[k] / sigma2[j])
        return np.array(v)

    # ── Step 3: Log marginal likelihood and optimisation ──────────────────────
    def neg_lml(lam):
        if lam <= 1e-6:
            return np.inf
        V0d    = build_V0_diag(lam)
        V0_inv = np.diag(1.0 / V0d)

        V1_inv = V0_inv + np.kron(Sig_inv, XtX)
        try:
            V1 = np.linalg.inv(V1_inv)
        except np.linalg.LinAlgError:
            return np.inf

        XtY_Sinv = X.T @ Y @ Sig_inv          # (K*p+1) x K
        m1       = V1 @ XtY_Sinv.T.ravel()    # K*(K*p+1) vector

        # Log marginal likelihood (constants absorbed)
        _, ld0 = np.linalg.slogdet(np.diag(V0d))
        _, ld1 = np.linalg.slogdet(V1)
        quad   = np.trace(Y.T @ Y @ Sig_inv) - m1 @ V1_inv @ m1
        lml    = 0.5 * ld1 - 0.5 * ld0 - 0.5 * quad
        return -lml

    result    = minimize_scalar(neg_lml, bounds=(0.001, 100), method="bounded")
    lam_opt   = result.x

    # ── Step 4: Posterior mean at optimal lambda ──────────────────────────────
    V0d       = build_V0_diag(lam_opt)
    V1_inv    = np.diag(1.0 / V0d) + np.kron(Sig_inv, XtX)
    V1        = np.linalg.inv(V1_inv)
    XtY_Sinv  = X.T @ Y @ Sig_inv
    m1        = V1 @ XtY_Sinv.T.ravel()

    B_post = m1.reshape(K, K * p + 1).T

    return B_post, lam_opt

# ── Helper: iterated forecast using B_post ───────────────────────────────────
def bvar_forecast(B_post, history, p, horizons):
    """
    Iterated multi-step forecast using posterior mean coefficient matrix.
    Identical structure to VAR iterated forecast — only coefficients differ.

    B_post : (K*p+1) x K  [intercept row | lag rows]
    Returns dict {h: log_real_ttf_forecast}
    """
    K   = history.shape[1]
    buf = [history[-(p - lag)] for lag in range(p)]   # [y_{t-p+1},...,y_t]

    fcsts = {}
    for h in range(1, max(horizons) + 1):
        # Build regressor: [1, y_{t+h-1}, y_{t+h-2}, ..., y_{t+h-p}]
        x    = np.concatenate([[1]] + [buf[-(lag)] for lag in range(1, p + 1)])
        yhat = x @ B_post                              # K-vector
        buf.append(yhat)
        if h in horizons:
            fcsts[h] = yhat[0]                         # log_real_ttf = var 0
    return fcsts

# ── Helper: AIC lag selection (same as VAR) ───────────────────────────────────
def select_aic_lag(data, max_lag):
    """Select lag order by AIC up to max_lag using OLS VAR fits."""
    best_aic, best_p = np.inf, 1
    for p in range(1, max_lag + 1):
        try:
            res = VAR(data).fit(p)
            if res.aic < best_aic:
                best_aic = res.aic
                best_p   = p
        except Exception:
            pass
    return best_p

# ── Main forecasting loop ─────────────────────────────────────────────────────
records      = []
eval_origins = df_var[df_var["date"] >= EVAL_START]["date"].tolist()

print(f"BVAR forecasting: {len(eval_origins)} origins x "
      f"{len(SPECIFICATIONS)} specs x {len(HORIZONS)} horizons")
print(f"Specifications:   {[s[0] for s in SPECIFICATIONS]}")
print(f"AIC cap:          {AIC_LAG_MAX} lags  (Baumeister et al. 2024)")
print(f"Prior:            Full multivariate GLP (Giannone et al. 2015)")
print()

for i, origin_date in enumerate(eval_origins):

    history = df_var[df_var["date"] <= origin_date][VARS].values
    T       = len(history)

    if i % 20 == 0:
        print(f"  Origin {i+1}/{len(eval_origins)}: "
              f"{origin_date.strftime('%Y-%m-%d')}  T={T}")

    for label, fixed_lag, use_aic in SPECIFICATIONS:

        # Lag order: AIC on OLS VAR
        p = select_aic_lag(history, AIC_LAG_MAX) if use_aic else fixed_lag

        if T <= p:
            continue

        try:
            B_post, lam_opt = bvar_glp(history, p)
            fcsts            = bvar_forecast(B_post, history, p, HORIZONS)
        except Exception:
            continue

        for h in HORIZONS:
            actual_ym = (origin_date + pd.DateOffset(months=h)).strftime("%Y-%m")
            records.append({
                "forecast_origin": origin_date.strftime("%Y-%m-%d"),
                "horizon":         h,
                "model":           label,
                "actual_month":    actual_ym,
                "forecast":        np.exp(fcsts[h]),
                "actual":          get_actual(actual_ym),
                "lag_order_used":  p,
                "lambda_opt":      round(lam_opt, 4),
            })

# ── Save output ───────────────────────────────────────────────────────────────
results = pd.DataFrame(records)
results.to_excel(OUTPUT_FILE, index=False)

# ── Summary ───────────────────────────────────────────────────────────────────
print()
print("=" * 60)
print("BVAR FORECASTING COMPLETE")
print("=" * 60)
print(f"  Total rows:       {len(results)}")
print(f"  Forecast origins: {results['forecast_origin'].nunique()}")
print(f"  Output:           {OUTPUT_FILE}")
print()

for label, _, _ in SPECIFICATIONS:
    sub      = results[results["model"] == label]
    origins  = sub["forecast_origin"].nunique()
    lag_dist = (sub.drop_duplicates("forecast_origin")["lag_order_used"]
                   .value_counts().sort_index().to_dict())
    lam_mean = sub.drop_duplicates("forecast_origin")["lambda_opt"].mean()
    lam_min  = sub.drop_duplicates("forecast_origin")["lambda_opt"].min()
    lam_max  = sub.drop_duplicates("forecast_origin")["lambda_opt"].max()
    print(f"  {label}:")
    print(f"    Origins:    {origins}")
    print(f"    Lag dist:   {lag_dist}")
    print(f"    Lambda:     mean={lam_mean:.3f}  min={lam_min:.3f}  max={lam_max:.3f}")
    print()

# Sample — first origin
first_origin = results["forecast_origin"].min()
sample = results[results["forecast_origin"] == first_origin]
print(f"Sample — first origin ({first_origin}):")
print(sample[["model","horizon","lag_order_used","lambda_opt",
              "actual_month","forecast","actual"]].to_string(index=False))


BVAR forecasting: 132 origins x 2 specs x 9 horizons
Specifications:   ['BVAR(1)', 'BVAR(AIC,p<=6)']
AIC cap:          6 lags  (Baumeister et al. 2024)
Prior:            Full multivariate GLP (Giannone et al. 2015)

  Origin 1/132: 2015-01-31  T=36
  Origin 21/132: 2016-09-30  T=56
  Origin 41/132: 2018-05-31  T=76
  Origin 61/132: 2020-01-31  T=96
  Origin 81/132: 2021-09-30  T=116
  Origin 101/132: 2023-05-31  T=136
  Origin 121/132: 2025-01-31  T=156

BVAR FORECASTING COMPLETE
  Total rows:       2376
  Forecast origins: 132
  Output:           Output_BVAR_forecasts.xlsx

  BVAR(1):
    Origins:    132
    Lag dist:   {1: 132}
    Lambda:     mean=16.552  min=2.426  max=44.011

  BVAR(AIC,p<=6):
    Origins:    132
    Lag dist:   {4: 61, 5: 19, 6: 52}
    Lambda:     mean=2.531  min=0.925  max=6.121

Sample — first origin (2015-01-31):
         model  horizon  lag_order_used  lambda_opt actual_month  forecast    actual
       BVAR(1)        1               1      2.9391      2015-0